In [1]:
import sympy as sym
import numpy as np
from matplotlib.pyplot import *
import math

In [2]:
x_P1O_R, y_P1O_R, x_P2O_R, y_P2O_R, t = sym.symbols('x_P1O_R y_P1O_R x_P2O_R y_P2O_R t')

In [3]:
# independent joint variables
th1 = sym.Function('th1')(t)
th2 = sym.Function('th2')(t)

In [4]:
# ax, ay, l1, l2, m1, m2, J_C1Z_L1, J_C2Z_L2, M_e1, M_e2, g__ = symbols('ax, ay, l1, l2, m1, m2, J_C1Z_L1, J_C2Z_L2, M_e1, M_e2, g')
# ax, ay ---> coordinates of joint Q
# l1, l2 ---> length of links
# m1, m2 ---> mass of links
# J_C1Z_L1, J_C2Z_L2 ---> moments of inertia of links wrt centre of mass about z-axis
# M_e1, M_e2 ---> external moments on joints
# g ---> gravitational acceleration
ax, ay = 0,0
l1, l2 = 0.1,0.1
m1, m2 = 0.3602528315,0.3602528315
M_e1, M_e2 = 0,0
g__ = 9.80665
J_C1Z_L1, J_C2Z_L2 = 2.3595417611E-05,2.3595417611E-05

In [5]:
# joint positions p1 and p2
p1 = sym.Matrix([[x_P1O_R],[y_P1O_R],[th1]])
p2 = sym.Matrix([[x_P2O_R],[y_P2O_R],[th2]])
p = sym.Matrix([[p1],[p2]])

In [9]:
# constraint position equation
g = sym.Matrix([
        [x_P1O_R - ax + l1 * sym.cos(th1)],
        [y_P1O_R - ay + l1 * sym.sin(th1)],
        [x_P2O_R - ax + l1 * sym.cos(th1) + l2 * sym.cos(th2)],
        [y_P2O_R - ay + l1 * sym.sin(th1) + l2 * sym.sin(th2)],
    ])
#pprint(g)

# constraint jacobian matrix
G = g.jacobian(p)
G

Matrix([
[1, 0, -0.1*sin(th1(t)), 0, 0,                0],
[0, 1,  0.1*cos(th1(t)), 0, 0,                0],
[0, 0, -0.1*sin(th1(t)), 1, 0, -0.1*sin(th2(t))],
[0, 0,  0.1*cos(th1(t)), 0, 1,  0.1*cos(th2(t))]])

In [10]:
# spatial inertia matrices of links
M1 = sym.Matrix([
        [m1, 0, -0.5 * m1 * l1 * sym.sin(th1)],
        [0, m1, 0.5 * m1 * l1 * sym.cos(th1)],
        [0.5 * m1 * l1 * sym.sin(th1), -0.5 * m1 * l1 * sym.cos(th1), J_C1Z_L1 + 0.25 * m1 * l1 * l1]
    ])
M2 = sym.Matrix([
        [m2, 0, -0.5 * m2 * l2 * sym.sin(th2)],
        [0, m2, 0.5 * m2 * l2 * sym.cos(th2)],
        [0.5 * m2 * l2 * sym.sin(th2), -0.5 * m2 * l2 * sym.cos(th2), J_C2Z_L2 + 0.25 * m2 * l2 * l2]
    ])
(m, n) = M1.shape
M = sym.Matrix([
    [M1, sym.zeros(m,n)],
    [sym.zeros(m,n), M2]
])
M

Matrix([
[              0.3602528315,                           0, -0.018012641575*sin(th1(t)),                          0,                           0,                           0],
[                         0,                0.3602528315,  0.018012641575*cos(th1(t)),                          0,                           0,                           0],
[0.018012641575*sin(th1(t)), -0.018012641575*cos(th1(t)),           0.000924227496361,                          0,                           0,                           0],
[                         0,                           0,                           0,               0.3602528315,                           0, -0.018012641575*sin(th2(t))],
[                         0,                           0,                           0,                          0,                0.3602528315,  0.018012641575*cos(th2(t))],
[                         0,                           0,                           0, 0.018012641575*sin(th2(t)), -0.018

In [11]:
# constraint acceleration equation: G . p'' = beta(p, p')
th1_ = th1.diff(t)
th2_ = th2.diff(t)
beta = sym.Matrix([
    [l1 * sym.cos(th1) * (th1_**2)],
    [l2 * sym.sin(th1) * (th2_**2)],
    [l1 * sym.cos(th1) * (th1_**2) + l2 * sym.cos(th2) * (th2_**2)],
    [l1 * sym.sin(th1) * (th1_**2) + l2 * sym.sin(th2) * (th2_**2)]
])
#pprint(beta)

In [12]:
# joint forces and torques
A_RL1 = sym.Matrix([
    [sym.cos(th1), -sym.sin(th1), 0],
    [sym.sin(th1), sym.cos(th1), 0],
    [0, 0, 1]
])

A_RL2 = sym.Matrix([
    [sym.cos(th2), -sym.sin(th2), 0],
    [sym.sin(th2), sym.cos(th2), 0],
    [0, 0, 1]
])

F_W1_R = sym.Matrix([[0],[-m1*g__],[0]])
F_W1_L1 = A_RL1.T * F_W1_R

F_W2_R = sym.Matrix([[0],[-m2*g__],[0]])
F_W2_L2 = A_RL2.T * F_W2_R 

M_W1_L1 = sym.Matrix([[0,0,0],[0,0,-l1/2],[0,l1/2,0]]) * F_W1_L1
M_W2_L2 = sym.Matrix([[0,0,0],[0,0,-l1/2],[0,l1/2,0]]) * F_W2_L2

M_e_L1 = sym.Matrix([[0], [0], [M_e1 - M_e2]])
M_e_L2 = sym.Matrix([[0], [0], [M_e2]])

In [13]:
f = sym.Matrix([
    [F_W1_R[0]],
    [F_W1_R[1]],
    [M_e_L1[2] + M_W1_L1[2]],
    [F_W2_R[0]],
    [F_W2_R[1]],
    [M_e_L2[2] + M_W2_L2[2]]
])

In [14]:
qG = sym.Matrix([
    [0.5 * m1 * l1 * sym.cos(th1) * th1_**2],
    [0.5 * m1 * l1 * sym.sin(th1) * th1_**2],
    [0],
    [0.5 * m2 * l2 * sym.cos(th2) * th2_**2],
    [0.5 * m2 * l2 * sym.sin(th2) * th2_**2],
    [0]
])

In [15]:
# vector of dependent joint coordinates
p_dep = sym.Matrix([[x_P1O_R], [y_P1O_R], [x_P2O_R], [y_P2O_R]])
#pprint(p_dep)

# vector of independent joint coordinates
p_ind = sym.Matrix([[th1], [th2]])

In [16]:
# explicit constraint position equation
h = sym.Matrix([
    [ax - l1 * sym.cos(th1)],
    [ay - l1 * sym.sin(th1)],
    [th1],
    [ax - l1 * sym.cos(th1) - l2 * sym.cos(th2)],
    [ay - l1 * sym.sin(th1) - l2 * sym.sin(th2)],
    [th2]
])

# explicit constraint jacobian matrix
h_pind = h.jacobian(p_ind)
#pprint(h_pind)

In [17]:
# Global projector J_v
T = sym.Identity(6)
T_ind = sym.Identity(2)
J_v = T.inv() * h_pind * T_ind
J_v

Matrix([
[ 0.1*sin(th1(t)),                0],
[-0.1*cos(th1(t)),                0],
[               1,                0],
[ 0.1*sin(th1(t)),  0.1*sin(th2(t))],
[-0.1*cos(th1(t)), -0.1*cos(th2(t))],
[               0,                1]])

In [18]:
J_v_dot = sym.Derivative(J_v, t, evaluate=True)
#pprint(J_v_dot)

In [20]:
M_ind = sym.simplify(J_v.T * M * J_v)
M_ind

Matrix([
[                   0.008129284126361, 0.0018012641575*cos(th1(t) - th2(t))],
[0.0054037924725*cos(th1(t) - th2(t)),                    0.004526755811361]])

In [22]:
v_ind = sym.Matrix([[th1_], [th2_]])
f_ind = sym.simplify(J_v.T * f - J_v.T * M * J_v_dot * v_ind)
f_ind

Matrix([
[-0.003602528315*sin(th1(t) - th2(t))*Derivative(th2(t), t)**2 + 0.529931014504421*cos(th1(t))],
[0.0054037924725*sin(th1(t) - th2(t))*Derivative(th1(t), t)**2 + 0.176643671501474*cos(th2(t))]])

In [23]:
qG_ind = sym.simplify(J_v.T * qG)
qG_ind

Matrix([
[0.0018012641575*sin(th1(t) - th2(t))*Derivative(th2(t), t)**2],
[                                                            0]])

In [24]:
theta_DDot = sym.simplify(M_ind.inv() * (f_ind + qG_ind))
theta_DDot

Matrix([
[  (6.62852315812439e-8*sin(th1(t) - th2(t))*Derivative(th2(t), t)**2 + 3.9563834496846e-8*sin(2*th1(t) - 2*th2(t))*Derivative(th1(t), t)**2 + 1.29329559189498e-6*cos(th1(t) - 2*th2(t)) - 1.82077863966912e-5*cos(th1(t)))/(7.91276689936921e-8*cos(th1(t) - th2(t))**2 - 2.99151836593167e-7)],
[(-4.39289643688433e-5*sin(th1(t) - th2(t))*Derivative(th1(t), t)**2 - 4.86682884764128e-6*sin(2*th1(t) - 2*th2(t))*Derivative(th2(t), t)**2 + 0.00143181861356164*cos(2*th1(t) - th2(t)) - 4.16798119741782e-6*cos(th2(t)))/(9.73365769528255e-6*cos(th1(t) - th2(t))**2 - 3.67992841612094e-5)]])

In [27]:
J_v

Matrix([
[ 0.1*sin(th1(t)),                0],
[-0.1*cos(th1(t)),                0],
[               1,                0],
[ 0.1*sin(th1(t)),  0.1*sin(th2(t))],
[-0.1*cos(th1(t)), -0.1*cos(th2(t))],
[               0,                1]])